# Interactive view of an axisymmetric case

Z3ST solves a body of revolution on its $(r, z)$ section. This notebook sweeps that section
about the axis and shows the cylinder in 3D, with a sector left open so the field through the
wall is visible on the cut faces.

It reads `output/fields.vtu` of the case chosen in the first cell: any folder of `cases/` or
`ndt-work/` that has been solved. A thermal case shows the temperature; a thermo-mechanical one
also the displacements and the stresses.

Needs `pip install "pyvista[jupyter]"` (already in the course environment).

In [1]:
# --- choose the case: every solved folder in cases/ and ndt-work/ is listed -------------
import glob, os
import numpy as np
import pyvista as pv
import ipywidgets as widgets

# 'html': the whole scene in the cell, rendered by the browser, no server: works in VS Code,
# Colab, Kaggle and Jupyter. 'trame' streams from a local server (Jupyter Lab only, often blank
# in VS Code); 'static' is a still picture, if nothing else shows.
pv.set_jupyter_backend(os.environ.get('NDT_PV_BACKEND', 'html'))

def _labs_root(start=None):
    # the course folder above this notebook: the one holding cases/
    d = os.path.abspath(start or os.getcwd())
    while not os.path.isdir(os.path.join(d, 'cases')):
        if os.path.dirname(d) == d:
            return os.getcwd()
        d = os.path.dirname(d)
    return d

LABS = _labs_root()
_found = sorted(glob.glob(os.path.join(LABS, 'cases', '*', 'output', 'fields.vtu'))) + \
         sorted(glob.glob(os.path.join(LABS, 'ndt-work', '*', 'output', 'fields.vtu')))
CASES = {os.path.relpath(os.path.dirname(os.path.dirname(f)), LABS): os.path.dirname(os.path.dirname(f))
         for f in _found}
if not CASES:
    raise SystemExit(f'no solved case under {LABS}: run one first (its output/fields.vtu is what is read)')

CASE = widgets.Dropdown(options=list(CASES),
                        value='cases/G4_lined_wall_thermal' if 'cases/G4_lined_wall_thermal' in CASES else list(CASES)[0],
                        description='case', layout=widgets.Layout(width='60%'))
CASE

Dropdown(description='case', layout=Layout(width='60%'), options=('cases/G4_lined_wall_thermal', 'cases/G5_lin…

In [2]:
# --- read the section, sweep it, draw it -------------------------------------------------
I_RR, I_TT, I_ZZ = 0, 4, 8          # z3st axisymmetric tensor order is (r, theta, z)

def load_section(case_dir):
    """The (r, z) section solved by z3st, placed in 3D as the half-plane y = 0, with the fields
    named and in readable units."""
    g = pv.read(os.path.join(case_dir, 'output', 'fields.vtu'))
    try:                                                         # the 2-D cells as PolyData
        s = g.extract_surface(algorithm='dataset_surface')       # pyvista >= 0.47
    except TypeError:
        s = g.extract_surface()                                  # older pyvista: the same filter
    sec = pv.PolyData(np.column_stack([s.points[:, 0], 0*s.points[:, 0], s.points[:, 1]]), s.faces)
    sec.point_data['T (degC)'] = np.asarray(s.point_data['Temperature']) - 273.15
    if 'Displacement' in s.point_data:
        U = np.asarray(s.point_data['Displacement'])
        sec.point_data['u_r (mm)'] = U[:, 0]*1e3
        sec.point_data['u_z (mm)'] = U[:, 1]*1e3
        S = np.asarray(s.point_data['Stress (points)'])
        sec.point_data['sigma_rr (MPa)'] = S[:, I_RR]/1e6
        sec.point_data['sigma_tt, hoop (MPa)'] = S[:, I_TT]/1e6
        sec.point_data['sigma_zz (MPa)'] = S[:, I_ZZ]/1e6
        sec.point_data['von Mises (MPa)'] = np.asarray(s.point_data['VonMises (points)'])/1e6
    return sec

FIELDS = ['T (degC)', 'u_r (mm)', 'u_z (mm)', 'sigma_rr (MPa)', 'sigma_tt, hoop (MPa)',
          'sigma_zz (MPa)', 'von Mises (MPa)']

def show(case=None, field='T (degC)', angle=270, z_scale=0.25, edges=False):
    """The case swept by `angle` degrees about the axis, the open sector facing the camera.
    `z_scale` shortens the height, so that the wall thickness is visible next to it."""
    case = case or CASE.value
    sec = load_section(CASES[case])
    if field not in sec.point_data:
        print(f'{field}: not in {case} (a thermal case has the temperature only)')
        return
    vol = sec.extrude_rotate(resolution=max(8, int(angle/4)), angle=angle, capping=True,
                             rotation_axis=(0, 0, 1)).rotate_z(360.0 - angle, inplace=False)
    vol.points[:, 2] *= z_scale
    p = pv.Plotter()
    p.add_mesh(vol, scalars=field, cmap='turbo', show_edges=edges,
               scalar_bar_args={'title': field, 'fmt': '%.4g'})
    p.add_text(f'{case}: {field}' + (f'   (height x{z_scale:g})' if z_scale != 1 else ''), font_size=10)
    p.view_isometric()
    p.show()

## Temperature

Stresses are point data: at the liner/vessel interface they average the two materials (G5, §8),
so read the value of the vessel there from the cell data, not from this picture.

In [3]:
show(field='T (degC)')

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…

## Explorer

Any case, field, sector and height scale. Each change rebuilds the view.

In [4]:
widgets.interact(
    show,
    case=CASE,
    field=widgets.Dropdown(options=FIELDS, value='T (degC)'),
    angle=widgets.IntSlider(min=90, max=360, step=15, value=270, description='sweep (deg)'),
    z_scale=widgets.FloatSlider(min=0.05, max=1.0, step=0.05, value=0.25, description='height x'),
    edges=False,
);

interactive(children=(Dropdown(description='case', layout=Layout(width='60%'), options=('cases/G4_lined_wall_t…